# Frozen-encoder ranking backends

Tests whether the recurrent-author problem comes from cosine geometry or candidate popularity. The encoder and source-heldout split stay frozen. Transform fitting uses train only; hyperparameters use dev only; test is read once for the final comparison.

Backends: cosine, centered cosine, all-but-top, shrinkage whitening, L1, Spearman, CSLS, adaptive S-Norm, regularized PLDA, and PLDA + S-Norm. Normalized Euclidean is omitted because it induces exactly the same ranking as cosine.

## Evidence and decision rule

Whitening addresses anisotropic sentence spaces (Su et al., 2021). CSLS and local scaling target hubs rather than manually penalizing named authors (Conneau et al., 2017; Schnitzer et al., 2012). S-Norm and PLDA are cohort-normalized and probabilistic scoring alternatives from speaker verification. Recent retrieval papers motivate, but do not establish, the 2026 hubness hypotheses for stylometry.

A backend passes only if its paired-profile MRR confidence interval is no worse than −0.01, Recall@3 falls by less than 0.01, and both false-top3 HHI and Gini improve over cosine. No passing backend means no production ranking change.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import importlib.util, json, os, subprocess, sys
missing = [name for name in ('pyarrow', 'sklearn', 'sentence_transformers') if importlib.util.find_spec(name) is None]
if missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pyarrow>=14', 'scikit-learn>=1.4', 'sentence-transformers>=4.1'], check=True)
import pandas as pd

REPO = Path('/content/drive/MyDrive/style_matching')
os.chdir(REPO)
GUTENBERG = REPO / 'artifacts/source_expansion_v2/gutenberg_targeted_v1'
HELDOUT = GUTENBERG / 'source_heldout_splits.parquet'
MODEL = REPO / 'artifacts/multilingual_author_style_v1'
EMBEDDINGS = GUTENBERG / 'frozen_encoder_eval'
OUT = GUTENBERG / 'similarity_backends_v1'
SEED = 20260902
assert HELDOUT.exists(), HELDOUT
assert (MODEL / 'model.safetensors').exists(), MODEL

def run(command):
    print('>>>', ' '.join(map(str, command)), flush=True)
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='')
    code = process.wait()
    if code:
        raise RuntimeError(f'command failed with exit {code}: {command}')


In [ ]:
# Encodes once. Reruns reuse aligned train/dev/test arrays.
run([
    sys.executable, 'scripts/style_embedding_recall.py',
    '--input', str(HELDOUT),
    '--out-dir', str(EMBEDDINGS),
    '--model-name', str(MODEL),
    '--batch-size', '128', '--train-cap', '300',
    '--eval-splits', 'dev,test', '--device', 'cuda',
    '--seed', str(SEED), '--skip-existing',
])

In [ ]:
# CPU matrix work after the embedding cell; GPU is not expected to stay busy.
run([
    sys.executable, 'scripts/evaluate_similarity_backends.py',
    '--input', str(HELDOUT),
    '--embedding-dir', str(EMBEDDINGS),
    '--output-dir', str(OUT),
    '--train-cap', '300', '--bootstrap-runs', '5000',
    '--seed', str(SEED),
])

In [ ]:
metrics = pd.read_csv(OUT / 'backend_test_metrics.csv')
display(metrics[[
    'method', 'selected_parameter', 'mrr', 'mrr_ci_low', 'mrr_ci_high',
    'recall_at_1', 'recall_at_3', 'recall_at_5',
    'false_top3_hhi', 'false_top3_gini',
    'worst_decile_profile_recall_at_3', 'adoption_gate',
]].sort_values('mrr', ascending=False).style.format(precision=4))
display(pd.read_csv(OUT / 'geometry_diagnostics.csv').style.format(precision=4))
display(pd.read_csv(OUT / 'backend_subgroup_metrics.csv').style.format(precision=4))

In [ ]:
report = json.loads((OUT / 'backend_metrics.json').read_text())
print('RECOMMENDED BACKEND:', report['recommended_backend'])
print('PRODUCTION CHANGE AUTHORIZED:', report['production_change_authorized'])
print('RETURN:', OUT / 'backend_metrics.json')
print('RETURN:', OUT / 'backend_test_metrics.csv')
print('RETURN:', OUT / 'backend_author_exposure.csv')
print('RETURN:', OUT / 'backend_subgroup_metrics.csv')
print('RETURN:', OUT / 'geometry_diagnostics.csv')